# 03 — LUNA16 classical ML (SVM / RF / KNN)

5-fold stratified outer CV with inner 3-fold grid search per model. Both default-threshold and Youden-calibrated metrics are saved per fold to `results/luna16_<model>.json`.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/luna16.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
print('X', X.shape, 'pos rate:', y.mean())

X (124, 91) pos rate: 0.9032258064516129


In [3]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'luna16_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])


=== SVM ===
[fold 0] train=99 test=25
[fold 0] best_params={'C': 10, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.979 AUC=0.957 BalAcc(cal)=0.630
[fold 1] train=99 test=25
[fold 1] best_params={'C': 1, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.338 AUC=0.957 BalAcc(cal)=0.500
[fold 2] train=99 test=25
[fold 2] best_params={'C': 0.1, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.493 AUC=1.000 BalAcc(cal)=1.000
[fold 3] train=99 test=25
[fold 3] best_params={'C': 10, 'gamma': 'scale', 'kernel': 'linear'} thr=0.703 AUC=1.000 BalAcc(cal)=1.000
[fold 4] train=100 test=24
[fold 4] best_params={'C': 1, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.292 AUC=1.000 BalAcc(cal)=0.750

=== RF ===
[fold 0] train=99 test=25
[fold 0] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100} thr=0.940 AUC=0.978 BalAcc(cal)=0.957
[fold 1] train=99 test=25
[fold 1] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100} thr=0.740 AUC=1.000 BalAcc(cal)=1.000
[fold 2] train=99 test=25
[fold 2] bes

In [4]:
table = format_results_table(aggregated)
table

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.840 ± 0.292,0.852 ± 0.331,0.700 ± 0.447,0.975 ± 0.036,0.870 ± 0.256,0.776 ± 0.223,0.983 ± 0.024,0.999 ± 0.002
RF,0.984 ± 0.036,0.983 ± 0.039,1.000 ± 0.000,1.000 ± 0.000,0.991 ± 0.020,0.991 ± 0.019,0.996 ± 0.010,0.999 ± 0.002
KNN,0.911 ± 0.035,0.910 ± 0.056,0.900 ± 0.224,0.992 ± 0.019,0.948 ± 0.022,0.905 ± 0.087,0.980 ± 0.023,0.997 ± 0.003


In [5]:
import json

for name, (_, grid) in CLASSICAL_REGISTRY.items():
    folds_path = results_dir / f'luna16_{name}.json'
    with folds_path.open() as f:
        raw = json.load(f)
    print(name, 'selected hparams per fold:')
    for r in raw:
        print(' ', r['fold'], r['best_params'])

SVM selected hparams per fold:
  0 {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
  1 {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
  2 {'C': 0.1, 'gamma': 0.1, 'kernel': 'rbf'}
  3 {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
  4 {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
RF selected hparams per fold:
  0 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
  1 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
  2 {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
  3 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
  4 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
KNN selected hparams per fold:
  0 {'n_neighbors': 3, 'p': 1, 'weights': 'distance'}
  1 {'n_neighbors': 9, 'p': 1, 'weights': 'uniform'}
  2 {'n_neighbors': 7, 'p': 1, 'weights': 'uniform'}
  3 {'n_neighbors': 21, 'p': 1, 'weights': 'distance'}
  4 {'n_neighbors': 9, 'p': 1, 'weights': 'distance'}
